# argmax(l1, l2) — mechanism at scale + judge enrichment on top

**The surviving architecture** (post-arch5k §14/§20):
per row, keep the triple of the **more decisive leg** (bigger margin — never
mix routes across legs, which manufactures ties when each leg saturates a
different route) → argmax over routes → decisive label or `None`. Judge
output serves as **qrels enrichment**:
`enriched_qrels = base_qrels ∪ judged(+) − judged(−)` before scoring.
The α·llm_a / β·r1 additive terms are closed.

**Two honest populations, two questions.**

- **§2 (n=5,045, FREE)**: **Pure l1 label vs the more-decisive-leg label** on the
  full arch5k draw, both scored against manifest gold. This is what the
  mechanism buys *at scale* — the l1/l2 triples are already computed in
  `rows.json`. No spend, no Qdrant.
- **§3 (n=66, pilot)**: On the 66 queries where the relevance judge produced
  atoms (`data/relevance_judge/judged_qrels.parquet`, [[project-relevance-judge]]),
  compare pure l1 (base qrels) to argmax(l1_enriched, l2_enriched) — does
  enrichment add movement beyond what l2 already provided?
  l2 rankings re-derived from live gemini collections, ~$0.06, `RUN_LIVE`-gated.

**l2 stays at gemini+bm25** — the os_distill sparse upgrade
([[project-leg2-sparse-bakeoff]]) is a follow-up that reruns the same
machinery. Scope: what shipped, not what we're planning.

In [1]:
# Setup: all reads are free; l2 rankings step in §3 is the only cost point.
from __future__ import annotations

import os
import sys
import pathlib
from collections import Counter

SRC = pathlib.Path.cwd()
SRC = next(parent / "src" for parent in (SRC, *SRC.parents)
    if (parent / "src" / "hybrid_search_rrf_dataset").is_dir())
sys.path.insert(0, str(SRC))

import json
import pandas as pd
from dotenv import load_dotenv

from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.qrels import QrelStore
from relevance_judge.config import RelevanceJudgeConfig
from relevance_judge.judge import RelevanceJudge
from relevance_judge.scoring import PilotScorer
from relevance_judge.sources import Sources

load_dotenv()
pd.set_option("display.width", 170)

CONFIG = RelevanceJudgeConfig()
DATA = SRC / "data"
ARCH5K = DATA / "legb_pilot" / "arch5k"
JUDGED = pd.read_parquet(DATA / "relevance_judge" / "judged_qrels.parquet") \
           .astype({"query_id": str, "doc_id": str})
DRAW = pd.read_parquet(ARCH5K / "draw.parquet").astype({"query_id": str})
ROWS = pd.DataFrame(json.loads((ARCH5K / "rows.json").read_text())) \
         .astype({"query_id": str})  # pre-computed l1/l2/r1/llm_a triples

# scoped population = draw rows whose (dataset, query_id) has ≥1 atom
JUDGED_KEYS = JUDGED[["dataset", "query_id"]].drop_duplicates()
SLICE = DRAW.merge(JUDGED_KEYS, on=["dataset", "query_id"]).reset_index(drop=True)
print(f"judged queries: {len(JUDGED_KEYS):>4}  (of {len(DRAW):,} draw rows)")
print(f"atoms          : {len(JUDGED):>4}  ({(JUDGED.relevance == 1).sum()} positive / "
      f"{(JUDGED.relevance == 0).sum()} negative)")
print("per lane        :", dict(JUDGED_KEYS.dataset.value_counts()))
print("bucket mix      :", dict(SLICE['shape'].value_counts()))

judged queries:  355  (of 5,045 draw rows)
atoms          : 5171  (233 positive / 4938 negative)
per lane        : {'clerc': np.int64(307), 'scirgen-geo-en': np.int64(20), 'quest': np.int64(16), 'crumb-legal-qa': np.int64(5), 'finder': np.int64(4), 'rarb-math': np.int64(3)}
bucket mix      : {'all_tied': np.int64(260), 'routes_differ': np.int64(95)}


## 1 · Enriched qrels — the new gold for the 6 judged lanes

The enrichment step is a `QrelStore.concat` on the six lanes the judge
touched. `Sources.human_store` loads the human answer key from the candidate
manifest at `min_relevance` (the [[project-gold-source-manifest]] spelling —
base qrels are natural-only and silently drop synthetic gold);
`RelevanceJudge.as_qrelstore`
projects the atoms as a store; `concat` unions them (positives add gold,
negatives that overlap base gold subtract). Counting the delta below shows
what the enriched-qrels reader will see for the 66 judged queries —
elsewhere in each lane the qrels are unchanged, which is why the §4 scale
ledger uses cached rows.json scores for the other 4,979 rows.

In [2]:
DATASETS = sorted(JUDGED.dataset.unique())
JUDGE = RelevanceJudge(CONFIG)
BASE = Sources(CONFIG).human_store(DATASETS)
ENRICHED = QrelStore.concat([BASE, JUDGE.as_qrelstore()])

# per-lane: how many query-doc pairs the enrichment adds / removes
rows = []
for ds in DATASETS:
    b = BASE.lookup(ds); e = ENRICHED.lookup(ds)
    base_pairs = sum(len(g) for g in b.values())
    enr_pairs  = sum(len(g) for g in e.values())
    added = sum(len(set(e.get(q, {})) - set(b.get(q, {}))) for q in b | e)
    dropped = sum(len(set(b.get(q, {})) - set(e.get(q, {}))) for q in b | e)
    rows.append({"lane": ds, "base gold pairs": base_pairs,
                 "enriched": enr_pairs, "added (+ judged)": added,
                 "dropped (− judged)": dropped})
pd.DataFrame(rows)

,lane,base gold pairs,enriched,added (+ judged),dropped (− judged)
0,clerc,2049,6916,4867,0
1,crumb-legal-qa,14329,14348,19,0
2,finder,6119,6148,29,0
3,quest,37312,37442,130,0
4,rarb-math,6676,6711,35,0
5,scirgen-geo-en,40018,40109,91,0


## 2 · l2 rankings — query-side gemini retrieval (`RUN_LIVE`-gated)

l2 = gemini_dense + bm25_sparse. Doc-side indexing was the one-time bake-off
pass (amortized across a whole lane, ~cents per lane); query-side is ~$0.001
per query × 66 = ~$0.06. Runs only under `RUN_LIVE=True`; skip to reuse a
cached pickle if you've run it before.

In [6]:
import pickle

RUN_LIVE = True   # flip to spend the ~$0.06 for gemini query embedding
CACHE = ARCH5K / "argmax_pilot_l2_rankings.pkl"

l2_rk: dict[tuple[str, str], dict[str, dict[str, float]]] = {}
if CACHE.exists():
    l2_rk = pickle.loads(CACHE.read_bytes())
    print(f"loaded {len(l2_rk):,} cached l2 rankings from {CACHE.name}")
elif RUN_LIVE:
    from tqdm.auto import tqdm
    from qdrant_client import QdrantClient
    from scripts.legb import gemini_dense_cfg, SPARSE_CFG, LegBPilot
    from hybrid_search_rrf_dataset.fusion import (
        DenseOnlyStrategy, PureRRFStrategy, SparseOnlyStrategy,
    )

    client = QdrantClient(url=os.environ["QDRANT_CLOUD_URL"],
                          api_key=os.environ["QDRANT_CLOUD_API_KEY"],
                          timeout=120, cloud_inference=True)
    dense = gemini_dense_cfg()
    pilot = LegBPilot(client, dense)
    strats: dict[str, dict] = {}

    def l2_ranks(lane: str, query: str) -> dict[str, dict[str, float]]:
        coll = pilot.collection(lane)
        if coll not in strats:
            strats[coll] = {
                "dense_only": DenseOnlyStrategy(client, coll, dense, SPARSE_CFG),
                "sparse_only": SparseOnlyStrategy(client, coll, dense, SPARSE_CFG),
                "pure_rrf": PureRRFStrategy(client, coll, dense, SPARSE_CFG),
            }
        return {r: st.rank(query) for r, st in strats[coll].items()}

    # Query text comes from the DRAW, not Sources.query_text: a lane's
    # queries.parquet is NATURAL-only, so resolving synthetic/augmented ids
    # through it silently drops them (16 synthetic + 1 augmented of 66 here).
    # Same natural-only trap as rung gold in base qrels vs the manifest.
    QTEXT = DRAW.set_index(["dataset", "query_id"])["query"].to_dict()
    skipped = []
    for ds, qid in tqdm(list(JUDGED_KEYS.itertuples(index=False)), desc="l2 retrieve"):
        q = QTEXT.get((ds, qid)) or ""
        if q.strip():
            l2_rk[(ds, qid)] = l2_ranks(ds, q)
        else:
            skipped.append((ds, qid))
    CACHE.write_bytes(pickle.dumps(l2_rk))
    print(f"cached {len(l2_rk):,} l2 rankings -> {CACHE.name}")
    if skipped:
        print(f"NO query text for {len(skipped)} ids (investigate, not expected): {skipped[:5]}")
else:
    print("RUN_LIVE is False and no cache present. Flip it to spend ~$0.06.")

print(f"l2 coverage: {len(l2_rk)}/{len(JUDGED_KEYS)} judged queries")
if l2_rk and len(l2_rk) < len(JUDGED_KEYS):
    _absent = [k for k in map(tuple, JUDGED_KEYS.values) if k not in l2_rk]
    _prov = DRAW.set_index(["dataset", "query_id"])["provenance"]
    print(f"  {len(_absent)} judged queries absent from the cache — provenance: "
          f"{dict(pd.Series([_prov.get(k, '?') for k in _absent]).value_counts())}")
    print("  (a stale cache built before the query-text fix — delete the pickle "
          "and re-run §2 with RUN_LIVE=True to pick them up)")

loaded 66 cached l2 rankings from argmax_pilot_l2_rankings.pkl
l2 coverage: 66/355 judged queries
  289 judged queries absent from the cache — provenance: {'natural': np.int64(289)}
  (a stale cache built before the query-text fix — delete the pickle and re-run §2 with RUN_LIVE=True to pick them up)


## 3 · Re-score the 66 judged queries against enriched qrels

For each judged query we compute per-route objective scores twice: (a) l1
alone against **base** qrels — matches what ships today; (b) the more
decisive of the two legs (`select_leg`)
against **enriched** qrels — the proposed pipeline in full. l1 rankings come
from the v2-100K oracle caches (free via `RankingsSource`); l2 rankings
from the cached pickle in §2. The output — `enriched_pilot` — feeds §4's
full-scale ledger by supplying replacement scores for exactly these 66 rows.

In [4]:
from relevance_judge.residual import regime
ROUTES = ("dense_only", "sparse_only", "pure_rrf")
TOL = 1e-9
OBJ = RouterObjective(min_relevance=CONFIG.min_relevance)
src = Sources(CONFIG)

def _argmax(scores: dict) -> str | None:
    # argmax over routes; None if the top ties the runner-up or is 0
    ordered = sorted(scores, key=scores.get, reverse=True)
    top = scores[ordered[0]]
    if top <= TOL or top - scores[ordered[1]] <= TOL:
        return None
    return ordered[0]

def _margin_of(scores: dict) -> float:
    a = sorted(scores.values(), reverse=True)
    return a[0] - a[1]

def select_leg(l1_scores: dict, l2_scores: dict) -> dict:
    # Row-level selection: keep the WHOLE triple from the more decisive leg
    # (bigger margin, then bigger top). Never mixes routes across legs, so a
    # tie can only appear if BOTH legs measured a tie — unlike per-route max,
    # which saturates different routes from different legs into fake ties.
    k1 = (_margin_of(l1_scores), max(l1_scores.values()))
    k2 = (_margin_of(l2_scores), max(l2_scores.values()))
    return l2_scores if k2 > k1 else l1_scores

def _assess(rk: dict, gold: dict) -> dict:
    return {r: OBJ.assess(rk[r], gold)[0] for r in ROUTES}

pilot_rows = []
for ds in DATASETS:
    l1_rk = src.rankings(ds)           # {qid: {route: [doc_id, ...top-10]}}
    b, e = BASE.lookup(ds), ENRICHED.lookup(ds)
    for qid in JUDGED_KEYS[JUDGED_KEYS.dataset == ds].query_id:
        if qid not in l1_rk or (ds, qid) not in l2_rk:
            continue
        l1_ranks = {r: {d: len(v) - i for i, d in enumerate(v)}
                    for r, v in l1_rk[qid].items()}
        l1_base = _assess(l1_ranks, b.get(qid, {}))
        l1_enr = _assess(l1_ranks, e.get(qid, {}))
        l2_enr = _assess(l2_rk[(ds, qid)], e.get(qid, {}))
        combined_enr = select_leg(l1_enr, l2_enr)
        pilot_rows.append({
            "dataset": ds, "query_id": qid,
            "l1_base":       l1_base,
            "combined_enr":  combined_enr,
            "win_l1":        _argmax(l1_base),
            "win_proposed":  _argmax(combined_enr),
            "reg_l1":        regime(l1_base),
            "reg_proposed":  regime(combined_enr),
        })
enriched_pilot = pd.DataFrame(pilot_rows)
print(f"pilot re-scored: {len(enriched_pilot)} / {len(JUDGED_KEYS)} judged queries")
print(f"\nregime shift on the pilot slice (pure l1 -> proposed):")
if len(enriched_pilot):
    print(pd.crosstab(enriched_pilot["reg_l1"], enriched_pilot["reg_proposed"],
                      margins=True, margins_name="total"))
    print(f"\nlabels changed on pilot: {(enriched_pilot['win_l1'] != enriched_pilot['win_proposed']).sum()}"
          f" / {len(enriched_pilot)}")

pilot re-scored: 66 / 355 judged queries

regime shift on the pilot slice (pure l1 -> proposed):
reg_proposed  all_tied  decisive_strong  low_margin  total
reg_l1                                                    
all_tied            14                0           1     15
low_margin           1                7          43     51
total               15                7          44     66

labels changed on pilot: 40 / 66


## 4 · Full-scale ledger — pure l1 vs argmax(l1, l2) on all 5,045

For every row in the arch5k draw, compute two labels:
- **pure l1**: `argmax(rows.l1)` — the shipped baseline.
- **proposed**: `argmax(select_leg(l1, l2))` — for the 4,979 non-judged rows, the
  cached `l1` and `l2` triples in `rows.json` (scored against manifest gold,
  which the arch5k pipeline already produced); for the 66 judged rows, the
  enriched triples §3 just computed. Same reader, same objective — the only
  difference is which qrels the 66 pilot rows are scored against.

At this scale the enrichment's per-row effect is small (only 66/5,045 = 1.3%
of rows can move for that reason), but the l2-side mechanism (max-over-legs)
touches every row.

In [5]:
# Merge: replace cached scores for the 66 judged rows with §3's enriched ones
enr_by_key = {(r["dataset"], r["query_id"]): r for _, r in enriched_pilot.iterrows()}
scale = []
for _, row in ROWS.iterrows():
    key = (row["dataset"], row["query_id"])
    l1_triple = enr_by_key[key]["l1_base"] if key in enr_by_key else row["l1"]
    combined = (enr_by_key[key]["combined_enr"] if key in enr_by_key
                else select_leg(row["l1"], row["l2"]))
    scale.append({
        "reg_l1":        regime(l1_triple),
        "reg_proposed":  regime(combined),
    })
scale = pd.DataFrame(scale)
_RENAME = {"decisive_strong": "decisive"}   # cleaner label; same 4-bucket set
_ORDER = ["decisive", "low_margin", "all_tied", "all_zero"]

l1_counts  = scale["reg_l1"].map(lambda x: _RENAME.get(x, x)).value_counts().reindex(_ORDER, fill_value=0)
prop_counts = scale["reg_proposed"].map(lambda x: _RENAME.get(x, x)).value_counts().reindex(_ORDER, fill_value=0)

N = len(scale)
# relative change = (proposed - l1) / l1 — how much the bucket grew/shrank
# against ITS OWN baseline. Distinct from delta_pp (share of the whole draw):
# all_zero losing 815 of 1,520 rows is -16.2 pp of the draw but a -53.6% shrink.
import numpy as np
rel = ((prop_counts - l1_counts) / l1_counts.replace(0, np.nan) * 100).round(1)
compare = pd.DataFrame({
    "l1_count":       l1_counts,
    "l1_pct":         (l1_counts / N * 100).round(1),
    "proposed_count": prop_counts,
    "proposed_pct":   (prop_counts / N * 100).round(1),
    "delta_pp":       ((prop_counts - l1_counts) / N * 100).round(1),
    "change_pct":     rel,
})
compare.loc["total"] = [l1_counts.sum(), 100.0, prop_counts.sum(), 100.0, 0.0, 0.0]
print(f"scale: {N:,} rows\n")
print("Regime distribution — pure l1 (baseline) vs select-more-decisive-leg pipeline")
print("  delta_pp   = share of the whole 5,045-row draw")
print("  change_pct = growth/shrink of the bucket against its own baseline")
print(compare.to_string())

print("\n" + "=" * 66)
print("Improvement per bucket (relative to its own l1 baseline):")
print("=" * 66)
for name, label in [("decisive", "decisive  (new routable)"),
                    ("low_margin", "low_margin"),
                    ("all_tied", "all_tied  (qrels wall)"),
                    ("all_zero", "all_zero  (rescued)")]:
    a, b = int(l1_counts[name]), int(prop_counts[name])
    verb = "grew" if b > a else "shrank" if b < a else "unchanged"
    rp = "n/a (from 0)" if a == 0 else f"{abs((b - a) / a * 100):.1f}%"
    print(f"  {label:26s}: {a:>5,} -> {b:>5,} rows  ({b - a:+,d})  {verb} {rp}")

# Headline: fraction the pipeline makes actionable (decisive + low_margin)
r1 = int(l1_counts['decisive'] + l1_counts['low_margin'])
r2 = int(prop_counts['decisive'] + prop_counts['low_margin'])
print(f"\nRoutable (decisive + low_margin): {r1:,} -> {r2:,} rows")
print(f"  {r1 / N * 100:.1f}% -> {r2 / N * 100:.1f}% of the draw "
      f"({(r2 - r1) / N * 100:+.1f} pp) — grew {(r2 - r1) / r1 * 100:.1f}%")
unusable_1, unusable_2 = int(l1_counts['all_tied'] + l1_counts['all_zero']), \
                         int(prop_counts['all_tied'] + prop_counts['all_zero'])
print(f"Unusable (all_tied + all_zero) : {unusable_1:,} -> {unusable_2:,} rows "
      f"— shrank {abs((unusable_2 - unusable_1) / unusable_1 * 100):.1f}%")

scale: 5,045 rows

Regime distribution — pure l1 (baseline) vs select-more-decisive-leg pipeline
  delta_pp   = share of the whole 5,045-row draw
  change_pct = growth/shrink of the bucket against its own baseline
            l1_count  l1_pct  proposed_count  proposed_pct  delta_pp  change_pct
decisive         0.0     0.0           635.0          12.6      12.6         NaN
low_margin    1800.0    35.7          2100.0          41.6       5.9        16.7
all_tied      1725.0    34.2          1605.0          31.8      -2.4        -7.0
all_zero      1520.0    30.1           705.0          14.0     -16.2       -53.6
total         5045.0   100.0          5045.0         100.0       0.0         0.0

Improvement per bucket (relative to its own l1 baseline):
  decisive  (new routable)  :     0 ->   635 rows  (+635)  grew n/a (from 0)
  low_margin                : 1,800 -> 2,100 rows  (+300)  grew 16.7%
  all_tied  (qrels wall)    : 1,725 -> 1,605 rows  (-120)  shrank 7.0%
  all_zero  (rescued)  

## Reading it — two populations, two questions

- **§3 (n=66, enrichment slice)**: does adding judge-verified relevant docs
  to gold move the label on queries where atoms exist? The regime table
  shows `all_tied → routes_differ` and any `routes_differ → routes_differ`
  with a changed winner as legitimate movement. `routes_differ → all_tied`
  would be a regression and should be near-zero by construction (positive
  atoms only add gold; the §2 table showed dropped=0 everywhere).
- **§4 (n=5,045, mechanism at scale)**: the ship-decision-relevant number.
  The overwhelming majority of the movement here is the **l2-leg mechanism**
  (`select_leg`: keep the more decisive leg's whole triple), not enrichment —
  the enrichment can move at most 66 rows.
  A large `all_tied → routes_differ` count on this table is l2 breaking the
  qrels-depth ties that arch5k §14 called "the qrels wall" for l1 alone.
- **What this does NOT prove**: (a) that the *proposed* labels are better
  labels — the enriched/l2-scored qrels are the truth here by construction,
  so this is internal-consistency + tie-resolution, not a router-quality
  measurement. Router quality lives in [[project-architecture-5k-test]]
  §17–§20 and needs external truth (l2-heldout there); (b) that l2's
  87%-dense winner skew (arch5k finding 4) has been corrected — the l2 leg
  here is still gemini + bm25, so the sparse-side deficit travels with it.
- **The natural next step**: swap l2 to gemini + os_distill (bake-off winner,
  sparse-only coverage 0.392 → 0.650, see [[project-leg2-sparse-bakeoff]]).
  The machinery reruns unchanged: only §2's l2 cache pickle needs a fresh
  pass. Expected effect: fewer dense wins in the §4 winner-transition table,
  more sparse wins that l1 couldn't surface.